In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 98
==================================================
Week: 14 of 24
Day: 98 of 168
Date: May 13, 2026 
Topic: Model Monitoring & Logging

Week 14 Progress:
✅ Day 92: Docker Fundamentals
✅ Day 93: Dockerize Security System
✅ Day 94: RESTful API Design & FastAPI Deep Dive
✅ Day 95: API Authentication & Security
✅ Day 96: Building a CI/CD Pipeline
✅ Day 97: Complete Test Suite & Performance Testing
🔄 Day 98: Model Monitoring & Logging (TODAY!)

Progress: 85.7% (6/7 days)

==================================================
🎯 Week 14 Project: MLOps & Production Engineering
- Containerize ML projects with Docker
- Build production-grade APIs with FastAPI
- Automate deployments with GitHub Actions CI/CD
- Add monitoring, logging, and error tracking

🎯 Today's Learning Objectives:
1. Implement structured logging with Loguru
2. Add Prometheus metrics to the FastAPI API
3. Set up Grafana dashboard for visualization
4. Implement alerting on model performance degradation
5. Complete Week 14 review and reflection

📚 Today's Structure:
   Part 1 (1.5h): Structured Logging with Loguru
   Part 2 (2h):   Prometheus Metrics
   Part 3 (1.5h): Grafana Dashboard Setup
   Part 4 (1h):   Week 14 Review & Reflection

🎯 SUCCESS CRITERIA:
   ✅ Structured JSON logs with Loguru for every API request
   ✅ Prometheus metrics: request count, latency, model inference time
   ✅ Grafana dashboard showing API health and model performance
   ✅ Alerts configured for error rate and latency thresholds
   ✅ Week 14 complete — full MLOps stack implemented

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
!{sys.executable} -m pip install loguru prometheus-fastapi-instrumentator prometheus-client -q

print("✅ Libraries installed!")

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

import os
import time
import json
from datetime import datetime

print("\n✅ All libraries imported successfully!")
print("=" * 80)

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


✅ Libraries installed!

📚 IMPORTING LIBRARIES

✅ All libraries imported successfully!


In [2]:
print("\n" + "=" * 80)
print("📝 PART 1: STRUCTURED LOGGING WITH LOGURU")
print("=" * 80)


📝 PART 1: STRUCTURED LOGGING WITH LOGURU


In [3]:
# ==================================================
# EXERCISE 1.1: WHY STRUCTURED LOGGING?
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Why Structured Logging?")
print("=" * 80)

"""
📖 THEORY: Structured Logging

Traditional logging outputs plain text:
  [2026-05-12 10:23:41] INFO: Detection completed in 850ms

Structured logging outputs JSON:
  {"time": "2026-05-12T10:23:41", "level": "INFO",
   "event": "detection_complete", "duration_ms": 850,
   "user_id": "u-123", "model": "yolov8n", "detections": 3}

Why JSON logs are better in production:
- Machine-readable: tools like Datadog, Grafana Loki can query them
- Filterable: find all requests where duration_ms > 1000
- Aggregatable: count detections per user per hour
- Alertable: trigger alert when error_rate > 5%
"""

print("""
📋 PLAIN TEXT vs STRUCTURED LOGGING:

  Plain text (hard to query):
  ─────────────────────────────────────────────────────────
  2026-05-12 10:23:41 INFO Request completed: POST /api/v1/detections 201 850ms
  2026-05-12 10:23:42 ERROR Detection failed: CUDA out of memory
  2026-05-12 10:23:43 INFO Request completed: GET /api/v1/detections 200 12ms

  Structured JSON (queryable, filterable):
  ─────────────────────────────────────────────────────────
  {"time":"2026-05-12T10:23:41Z","level":"INFO","event":"request_complete",
   "method":"POST","path":"/api/v1/detections","status":201,
   "duration_ms":850,"user_id":"u-123","request_id":"a1b2c3"}

  {"time":"2026-05-12T10:23:42Z","level":"ERROR","event":"detection_failed",
   "error":"CUDA out of memory","model":"yolov8x","request_id":"d4e5f6"}

  {"time":"2026-05-12T10:23:43Z","level":"INFO","event":"request_complete",
   "method":"GET","path":"/api/v1/detections","status":200,
   "duration_ms":12,"user_id":"u-456","request_id":"g7h8i9"}

  With structured logs you can ask:
  → "Show all requests slower than 500ms in the last hour"
  → "How many detections did user u-123 run today?"
  → "What's the error rate for POST /api/v1/detections?"
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Why Structured Logging?

📋 PLAIN TEXT vs STRUCTURED LOGGING:

  Plain text (hard to query):
  ─────────────────────────────────────────────────────────
  2026-05-12 10:23:41 INFO Request completed: POST /api/v1/detections 201 850ms
  2026-05-12 10:23:42 ERROR Detection failed: CUDA out of memory
  2026-05-12 10:23:43 INFO Request completed: GET /api/v1/detections 200 12ms

  Structured JSON (queryable, filterable):
  ─────────────────────────────────────────────────────────
  {"time":"2026-05-12T10:23:41Z","level":"INFO","event":"request_complete",
   "method":"POST","path":"/api/v1/detections","status":201,
   "duration_ms":850,"user_id":"u-123","request_id":"a1b2c3"}

  {"time":"2026-05-12T10:23:42Z","level":"ERROR","event":"detection_failed",
   "error":"CUDA out of memory","model":"yolov8x","request_id":"d4e5f6"}

  {"time":"2026-05-12T10:23:43Z","level":"INFO","event":"request_complete",
   "method":"GET","path":"/api/v1/detections","status":200,
   "duration_ms":12

In [4]:
# ==================================================
# EXERCISE 1.2: LOGURU SETUP
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Loguru Setup for FastAPI")
print("=" * 80)

"""
📖 THEORY: Loguru

Loguru is a Python logging library that's much simpler than
the built-in logging module. Key features:
- No handlers, formatters, or loggers to configure
- Structured JSON output out of the box
- Automatic exception tracing with pretty formatting
- File rotation and retention built-in
- Works alongside uvicorn's logging
"""

logger_py = '''
# ============================================================
# api/core/logger.py — Loguru Configuration
# ============================================================

import sys
import json
from loguru import logger
from api.core.config import settings


def serialize(record) -> str:
    """
    Custom JSON serializer for structured log output.
    Called for every log record when using JSON sink.
    """
    subset = {
        "time": record["time"].strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
        "level": record["level"].name,
        "event": record["message"],
        "module": record["module"],
        "function": record["function"],
        "line": record["line"],
    }
    # Include any extra fields passed via logger.bind() or logger.opt()
    subset.update(record["extra"])

    # Include exception info if present
    if record["exception"]:
        subset["exception"] = str(record["exception"])

    return json.dumps(subset)


def setup_logging():
    """Configure Loguru — call once at application startup."""

    # Remove default handler
    logger.remove()

    if settings.LOG_LEVEL == "DEBUG" or settings.ENV == "development":
        # Development: human-readable colored output
        logger.add(
            sys.stdout,
            level=settings.LOG_LEVEL,
            format=(
                "<green>{time:HH:mm:ss}</green> | "
                "<level>{level: <8}</level> | "
                "<cyan>{module}</cyan>:<cyan>{line}</cyan> | "
                "{message}"
            ),
            colorize=True,
        )
    else:
        # Production: structured JSON output (machine-readable)
        logger.add(
            sys.stdout,
            level=settings.LOG_LEVEL,
            format="{message}",
            serialize=True,   # Built-in JSON serialization
        )

    # Always write to rotating log file
    logger.add(
        "logs/app_{time:YYYY-MM-DD}.log",
        level="INFO",
        rotation="1 day",          # New file every day
        retention="30 days",       # Keep 30 days of logs
        compression="zip",         # Compress old logs
        serialize=True,            # JSON format in files
    )

    # Separate error log file
    logger.add(
        "logs/error_{time:YYYY-MM-DD}.log",
        level="ERROR",
        rotation="1 week",
        retention="90 days",
        compression="zip",
        serialize=True,
        backtrace=True,            # Full traceback for errors
        diagnose=True,             # Variable values in traceback
    )

    return logger


# Initialize on import
setup_logging()
'''

loguru_usage_py = '''
# ============================================================
# How to use Loguru throughout the API
# ============================================================

from api.core.logger import logger


# ── Basic logging ─────────────────────────────────────────────
logger.info("Server starting up")
logger.warning("Model confidence below threshold")
logger.error("Redis connection failed")
logger.debug("Processing frame 42 of 100")


# ── Structured logging with extra fields ──────────────────────
# bind() adds persistent context to all subsequent log calls
request_logger = logger.bind(
    request_id="a1b2c3",
    user_id="u-123",
    endpoint="/api/v1/detections"
)
request_logger.info("Detection started")
request_logger.info("Detection complete", duration_ms=850, detections=3)


# ── Exception logging ──────────────────────────────────────────
try:
    result = model(frame)
except Exception as e:
    logger.exception(f"Model inference failed: {e}")
    # logger.exception() automatically captures the full traceback


# ── Performance logging ────────────────────────────────────────
import time

start = time.perf_counter()
result = model(frame)
duration_ms = round((time.perf_counter() - start) * 1000, 2)

logger.info(
    "inference_complete",
    duration_ms=duration_ms,
    num_detections=len(result[0].boxes),
    model_version="yolov8n",
    confidence_threshold=0.5,
)


# ── In middleware (structured per-request logging) ─────────────
# api/core/middleware.py (updated)

async def dispatch(self, request: Request, call_next):
    request_id = str(uuid.uuid4())[:8]
    start_time = time.perf_counter()

    # Bind request context to all logs within this request
    with logger.contextualize(
        request_id=request_id,
        method=request.method,
        path=request.url.path,
        client_ip=request.client.host,
    ):
        logger.info("request_received")

        response = await call_next(request)

        duration_ms = round((time.perf_counter() - start_time) * 1000, 2)
        logger.info(
            "request_complete",
            status_code=response.status_code,
            duration_ms=duration_ms,
        )

    return response
'''

print("📄 api/core/logger.py:")
print(logger_py)
print("\n📄 Loguru usage examples:")
print(loguru_usage_py)

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Loguru Setup for FastAPI
📄 api/core/logger.py:

# ============================================================
# api/core/logger.py — Loguru Configuration
# ============================================================

import sys
import json
from loguru import logger
from api.core.config import settings


def serialize(record) -> str:
    """
    Custom JSON serializer for structured log output.
    Called for every log record when using JSON sink.
    """
    subset = {
        "time": record["time"].strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
        "level": record["level"].name,
        "event": record["message"],
        "module": record["module"],
        "function": record["function"],
        "line": record["line"],
    }
    # Include any extra fields passed via logger.bind() or logger.opt()
    subset.update(record["extra"])

    # Include exception info if present
    if record["exception"]:
        subset["exception"] = str(record["exception"])

    return json.dumps(

In [5]:
print("\n" + "=" * 80)
print("📊 PART 2: PROMETHEUS METRICS")
print("=" * 80)


📊 PART 2: PROMETHEUS METRICS


In [6]:
# ==================================================
# EXERCISE 2.1: PROMETHEUS FUNDAMENTALS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Prometheus Fundamentals")
print("=" * 80)

"""
📖 THEORY: Prometheus

Prometheus is an open-source monitoring system that:
1. Scrapes metrics from your app at regular intervals (every 15s)
2. Stores them as time-series data
3. Lets you query with PromQL (Prometheus Query Language)
4. Feeds data to Grafana for visualization

Your app exposes a /metrics endpoint.
Prometheus polls /metrics → stores → Grafana visualizes.

Metric types:
- Counter    → always goes up (total requests, total errors)
- Gauge      → can go up or down (current active connections, memory usage)
- Histogram  → tracks distribution (request latency, response size)
- Summary    → like histogram but calculates quantiles client-side
"""

print("""
📋 PROMETHEUS METRIC TYPES:

  COUNTER (always increasing):
  ─────────────────────────────────────────────────────────
  http_requests_total{method="POST", status="201"}  → 1547
  detection_errors_total{reason="cuda_oom"}         → 3
  Use: count events that never decrease

  GAUGE (current value, can go up or down):
  ─────────────────────────────────────────────────────────
  active_connections                    → 12
  model_memory_usage_bytes              → 2147483648
  redis_cache_size                      → 847
  Use: measure current state

  HISTOGRAM (distribution with buckets):
  ─────────────────────────────────────────────────────────
  http_request_duration_seconds_bucket{le="0.1"}  → 1203
  http_request_duration_seconds_bucket{le="0.5"}  → 1489
  http_request_duration_seconds_bucket{le="1.0"}  → 1531
  http_request_duration_seconds_bucket{le="+Inf"} → 1547
  http_request_duration_seconds_sum               → 423.7
  http_request_duration_seconds_count             → 1547
  Use: measure latency, response sizes (p50, p95, p99 from buckets)

📋 KEY PROMQL QUERIES:

  # Request rate (requests/second over last 5 minutes)
  rate(http_requests_total[5m])

  # Error rate (% of requests that are 5xx)
  rate(http_requests_total{status=~"5.."}[5m])
  / rate(http_requests_total[5m]) * 100

  # p95 latency
  histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m]))

  # Average inference time
  rate(inference_duration_seconds_sum[5m])
  / rate(inference_duration_seconds_count[5m])
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Prometheus Fundamentals

📋 PROMETHEUS METRIC TYPES:

  COUNTER (always increasing):
  ─────────────────────────────────────────────────────────
  http_requests_total{method="POST", status="201"}  → 1547
  detection_errors_total{reason="cuda_oom"}         → 3
  Use: count events that never decrease

  GAUGE (current value, can go up or down):
  ─────────────────────────────────────────────────────────
  active_connections                    → 12
  model_memory_usage_bytes              → 2147483648
  redis_cache_size                      → 847
  Use: measure current state

  HISTOGRAM (distribution with buckets):
  ─────────────────────────────────────────────────────────
  http_request_duration_seconds_bucket{le="0.1"}  → 1203
  http_request_duration_seconds_bucket{le="0.5"}  → 1489
  http_request_duration_seconds_bucket{le="1.0"}  → 1531
  http_request_duration_seconds_bucket{le="+Inf"} → 1547
  http_request_duration_seconds_sum               → 423.7
  http_request_durat

In [7]:
# ==================================================
# EXERCISE 2.2: PROMETHEUS METRICS IN FASTAPI
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Adding Prometheus Metrics to FastAPI")
print("=" * 80)

metrics_py = '''
# ============================================================
# api/core/metrics.py — Prometheus Metrics
# ============================================================

from prometheus_client import Counter, Gauge, Histogram, Summary
from prometheus_fastapi_instrumentator import Instrumentator
from fastapi import FastAPI


# ── Custom ML-specific metrics ────────────────────────────────

# Total inference requests
INFERENCE_REQUESTS = Counter(
    name="ml_inference_requests_total",
    documentation="Total number of ML inference requests",
    labelnames=["model_version", "status"],  # labels = dimensions to filter by
)

# Inference duration histogram
INFERENCE_DURATION = Histogram(
    name="ml_inference_duration_seconds",
    documentation="Time taken for ML model inference",
    labelnames=["model_version"],
    buckets=[0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0],  # seconds
)

# Detections per request
DETECTIONS_PER_REQUEST = Histogram(
    name="ml_detections_per_request",
    documentation="Number of objects detected per inference request",
    labelnames=["class_name"],
    buckets=[0, 1, 2, 5, 10, 20, 50, 100],
)

# Confidence score distribution
DETECTION_CONFIDENCE = Histogram(
    name="ml_detection_confidence",
    documentation="Distribution of detection confidence scores",
    labelnames=["class_name"],
    buckets=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
)

# Current model in memory
MODEL_LOADED = Gauge(
    name="ml_model_loaded",
    documentation="Whether the ML model is loaded (1=yes, 0=no)",
)

# Redis cache stats
CACHE_HITS = Counter(
    name="redis_cache_hits_total",
    documentation="Total Redis cache hits",
)
CACHE_MISSES = Counter(
    name="redis_cache_misses_total",
    documentation="Total Redis cache misses",
)

# Active detection jobs
ACTIVE_INFERENCES = Gauge(
    name="ml_active_inferences",
    documentation="Number of inference requests currently processing",
)


# ── Auto-instrumentation (HTTP metrics) ───────────────────────
def setup_metrics(app: FastAPI):
    """
    Sets up automatic HTTP metrics (requests, latency, errors)
    AND exposes the /metrics endpoint for Prometheus to scrape.
    """
    Instrumentator(
        should_group_status_codes=False,
        should_ignore_untemplated=True,
        should_respect_env_var_absence=True,
        should_instrument_requests_inprogress=True,
        excluded_handlers=["/metrics", "/health"],
    ).instrument(app).expose(app, endpoint="/metrics")

    # Mark model as loaded at startup
    MODEL_LOADED.set(1)


# ── Usage in detection service ────────────────────────────────
# async def run_detection(self, file, confidence):
#     ACTIVE_INFERENCES.inc()          # Gauge up
#     try:
#         with INFERENCE_DURATION.labels(
#             model_version="yolov8n"
#         ).time():                     # Auto-times the block
#             result = self._run_yolo_inference(contents, confidence)
#
#         INFERENCE_REQUESTS.labels(
#             model_version="yolov8n",
#             status="success"
#         ).inc()
#
#         for det in result:
#             DETECTIONS_PER_REQUEST.labels(
#                 class_name=det["class_name"]
#             ).observe(1)
#             DETECTION_CONFIDENCE.labels(
#                 class_name=det["class_name"]
#             ).observe(det["confidence"])
#
#     except Exception as e:
#         INFERENCE_REQUESTS.labels(
#             model_version="yolov8n",
#             status="error"
#         ).inc()
#         raise
#     finally:
#         ACTIVE_INFERENCES.dec()      # Gauge down (always)
'''

print("📄 api/core/metrics.py:")
print(metrics_py)

print("""
💡 KEY METRIC DESIGN RULES:

  Labels are dimensions — use them to slice data:
  ✅ INFERENCE_REQUESTS.labels(model_version="yolov8n", status="success")
  ✅ INFERENCE_DURATION.labels(model_version="yolov8n").observe(0.85)

  Keep label cardinality LOW:
  ❌ labels(user_id="u-12345")  → millions of users = millions of series
  ✅ labels(role="operator")    → few roles = few series

  Always use .time() context manager for timing:
  with INFERENCE_DURATION.labels(model_version="yolov8n").time():
      result = model(frame)   ← duration auto-recorded on exit

  Always decrement Gauges in finally blocks:
  try:
      ACTIVE_INFERENCES.inc()
      result = run_inference()
  finally:
      ACTIVE_INFERENCES.dec()   ← runs even if exception raised
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Adding Prometheus Metrics to FastAPI
📄 api/core/metrics.py:

# ============================================================
# api/core/metrics.py — Prometheus Metrics
# ============================================================

from prometheus_client import Counter, Gauge, Histogram, Summary
from prometheus_fastapi_instrumentator import Instrumentator
from fastapi import FastAPI


# ── Custom ML-specific metrics ────────────────────────────────

# Total inference requests
INFERENCE_REQUESTS = Counter(
    name="ml_inference_requests_total",
    documentation="Total number of ML inference requests",
    labelnames=["model_version", "status"],  # labels = dimensions to filter by
)

# Inference duration histogram
INFERENCE_DURATION = Histogram(
    name="ml_inference_duration_seconds",
    documentation="Time taken for ML model inference",
    labelnames=["model_version"],
    buckets=[0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0],  # seconds
)

# Detections per request
DET

In [8]:
print("\n" + "=" * 80)
print("📈 PART 3: GRAFANA DASHBOARD SETUP")
print("=" * 80)


📈 PART 3: GRAFANA DASHBOARD SETUP


In [9]:
# ==================================================
# EXERCISE 3.1: MONITORING STACK WITH DOCKER COMPOSE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Full Monitoring Stack with Docker Compose")
print("=" * 80)

"""
📖 THEORY: The Monitoring Stack

The standard open-source monitoring stack for ML APIs:
  App → exposes /metrics
  Prometheus → scrapes /metrics every 15s, stores time-series
  Grafana → queries Prometheus, displays dashboards
  Alertmanager → receives alerts from Prometheus, sends notifications
"""

docker_compose_monitoring = '''
# ============================================================
# docker-compose.monitoring.yml
# Add to the Security System stack with:
# docker compose -f docker-compose.yml -f docker-compose.monitoring.yml up
# ============================================================

version: "3.9"

services:

  # ── Prometheus ────────────────────────────────────────────
  prometheus:
    image: prom/prometheus:v2.51.0
    container_name: security_prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - ./monitoring/alerts.yml:/etc/prometheus/alerts.yml:ro
      - prometheus_data:/prometheus
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.retention.time=30d"
      - "--web.enable-lifecycle"        # Allow config reload without restart
    restart: unless-stopped
    networks:
      - security_network

  # ── Grafana ───────────────────────────────────────────────
  grafana:
    image: grafana/grafana:10.4.0
    container_name: security_grafana
    ports:
      - "3000:3000"
    volumes:
      - grafana_data:/var/lib/grafana
      - ./monitoring/grafana/dashboards:/etc/grafana/provisioning/dashboards:ro
      - ./monitoring/grafana/datasources:/etc/grafana/provisioning/datasources:ro
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=${GRAFANA_PASSWORD:-admin}
      - GF_USERS_ALLOW_SIGN_UP=false
      - GF_ANALYTICS_REPORTING_ENABLED=false
    depends_on:
      - prometheus
    restart: unless-stopped
    networks:
      - security_network

  # ── Alertmanager ──────────────────────────────────────────
  alertmanager:
    image: prom/alertmanager:v0.27.0
    container_name: security_alertmanager
    ports:
      - "9093:9093"
    volumes:
      - ./monitoring/alertmanager.yml:/etc/alertmanager/alertmanager.yml:ro
    restart: unless-stopped
    networks:
      - security_network

volumes:
  prometheus_data:
  grafana_data:
'''

prometheus_yml = '''
# ============================================================
# monitoring/prometheus.yml — Prometheus Configuration
# ============================================================

global:
  scrape_interval: 15s          # Scrape every 15 seconds
  evaluation_interval: 15s      # Evaluate alert rules every 15s

# Alert rules file
rule_files:
  - "alerts.yml"

# Alertmanager connection
alerting:
  alertmanagers:
    - static_configs:
        - targets: ["alertmanager:9093"]

# Scrape targets
scrape_configs:

  # Scrape the FastAPI app metrics
  - job_name: "security_api"
    static_configs:
      - targets: ["fastapi_backend:8000"]
    metrics_path: "/metrics"
    scrape_interval: 15s

  # Scrape Prometheus itself
  - job_name: "prometheus"
    static_configs:
      - targets: ["localhost:9090"]

  # Scrape Redis exporter (optional — gives Redis memory/hit rate metrics)
  - job_name: "redis"
    static_configs:
      - targets: ["redis_cache:6379"]
'''

print("📄 docker-compose.monitoring.yml:")
print(docker_compose_monitoring)
print("\n📄 monitoring/prometheus.yml:")
print(prometheus_yml)

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Full Monitoring Stack with Docker Compose
📄 docker-compose.monitoring.yml:

# ============================================================
# docker-compose.monitoring.yml
# Add to the Security System stack with:
# docker compose -f docker-compose.yml -f docker-compose.monitoring.yml up
# ============================================================

version: "3.9"

services:

  # ── Prometheus ────────────────────────────────────────────
  prometheus:
    image: prom/prometheus:v2.51.0
    container_name: security_prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - ./monitoring/alerts.yml:/etc/prometheus/alerts.yml:ro
      - prometheus_data:/prometheus
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.retention.time=30d"
      - "--web.enable-lifecycle"        # Allow config reload without restart
    restart: unless-stopped
    networks:
      - sec

In [10]:
# ==================================================
# EXERCISE 3.2: ALERTS AND GRAFANA DASHBOARD
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Alerts & Grafana Dashboard Config")
print("=" * 80)

alerts_yml = '''
# ============================================================
# monitoring/alerts.yml — Prometheus Alert Rules
# ============================================================

groups:
  - name: security_system_alerts
    rules:

      # ── API Health ──────────────────────────────────────────
      - alert: HighErrorRate
        expr: |
          rate(http_requests_total{status=~"5.."}[5m])
          / rate(http_requests_total[5m]) * 100 > 5
        for: 2m                       # Must be true for 2 min before firing
        labels:
          severity: critical
        annotations:
          summary: "High API error rate"
          description: "Error rate is {{ $value | printf \\"%.1f\\" }}% (threshold: 5%)"

      - alert: HighLatency
        expr: |
          histogram_quantile(0.95,
            rate(http_request_duration_seconds_bucket[5m])
          ) > 3.0
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High API latency"
          description: "p95 latency is {{ $value | printf \\"%.2f\\" }}s (threshold: 3s)"

      # ── ML Model ────────────────────────────────────────────
      - alert: ModelNotLoaded
        expr: ml_model_loaded == 0
        for: 1m
        labels:
          severity: critical
        annotations:
          summary: "ML model not loaded"
          description: "The YOLO model is not loaded — all inference requests will fail"

      - alert: SlowInference
        expr: |
          histogram_quantile(0.95,
            rate(ml_inference_duration_seconds_bucket[10m])
          ) > 5.0
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "Slow ML inference"
          description: "p95 inference time is {{ $value | printf \\"%.2f\\" }}s"

      - alert: HighInferenceErrorRate
        expr: |
          rate(ml_inference_requests_total{status="error"}[5m])
          / rate(ml_inference_requests_total[5m]) * 100 > 10
        for: 3m
        labels:
          severity: critical
        annotations:
          summary: "High ML inference error rate"
          description: "{{ $value | printf \\"%.1f\\" }}% of inferences are failing"

      # ── Cache ────────────────────────────────────────────────
      - alert: LowCacheHitRate
        expr: |
          rate(redis_cache_hits_total[10m])
          / (rate(redis_cache_hits_total[10m]) + rate(redis_cache_misses_total[10m]))
          * 100 < 20
        for: 10m
        labels:
          severity: warning
        annotations:
          summary: "Low Redis cache hit rate"
          description: "Cache hit rate is {{ $value | printf \\"%.1f\\" }}% (target: >20%)"
'''

grafana_dashboard = '''
# ============================================================
# Grafana Dashboard Panels — AI Security System
# (Summary — full JSON provisioned in monitoring/grafana/dashboards/)
# ============================================================

"""
Dashboard: AI Security System Overview
Refresh: 30s
Time range: Last 1 hour

Row 1: API Health
─────────────────────────────────────────────────────────────
[Stat]   Total Requests (last 1h)
         sum(increase(http_requests_total[1h]))

[Stat]   Error Rate %
         rate(http_requests_total{status=~"5.."}[5m])
         / rate(http_requests_total[5m]) * 100

[Stat]   p95 Latency
         histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m]))

[Stat]   Active Inferences
         ml_active_inferences

Row 2: ML Model Performance
─────────────────────────────────────────────────────────────
[Graph]  Inference Duration (p50, p95, p99)
         histogram_quantile(0.50, rate(ml_inference_duration_seconds_bucket[5m]))
         histogram_quantile(0.95, rate(ml_inference_duration_seconds_bucket[5m]))
         histogram_quantile(0.99, rate(ml_inference_duration_seconds_bucket[5m]))

[Graph]  Detections per Minute by Class
         sum by (class_name)(rate(ml_detections_per_request[1m])) * 60

[Graph]  Confidence Score Distribution
         histogram_quantile(0.5, rate(ml_detection_confidence_bucket[5m]))

Row 3: Cache Performance
─────────────────────────────────────────────────────────────
[Stat]   Cache Hit Rate %
         rate(redis_cache_hits_total[5m])
         / (rate(redis_cache_hits_total[5m]) + rate(redis_cache_misses_total[5m]))
         * 100

[Graph]  Cache Hits vs Misses over time
         rate(redis_cache_hits_total[1m])
         rate(redis_cache_misses_total[1m])

Access Grafana at: http://localhost:3000
Default login: admin / (set via GRAFANA_PASSWORD env var)
"""
'''

print("📄 monitoring/alerts.yml:")
print(alerts_yml)
print("\n📄 Grafana Dashboard Summary:")
print(grafana_dashboard)

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Alerts & Grafana Dashboard Config
📄 monitoring/alerts.yml:

# ============================================================
# monitoring/alerts.yml — Prometheus Alert Rules
# ============================================================

groups:
  - name: security_system_alerts
    rules:

      # ── API Health ──────────────────────────────────────────
      - alert: HighErrorRate
        expr: |
          rate(http_requests_total{status=~"5.."}[5m])
          / rate(http_requests_total[5m]) * 100 > 5
        for: 2m                       # Must be true for 2 min before firing
        labels:
          severity: critical
        annotations:
          summary: "High API error rate"
          description: "Error rate is {{ $value | printf \"%.1f\" }}% (threshold: 5%)"

      - alert: HighLatency
        expr: |
          histogram_quantile(0.95,
            rate(http_request_duration_seconds_bucket[5m])
          ) > 3.0
        for: 5m
        labels:
          severity: 

In [11]:
print("\n" + "=" * 80)
print("🎓 PART 4: WEEK 14 REVIEW & REFLECTION")
print("=" * 80)


🎓 PART 4: WEEK 14 REVIEW & REFLECTION


In [12]:
# ==================================================
# EXERCISE 4.1: WEEK 14 COMPLETE REVIEW
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.1: Week 14 Complete Review")
print("=" * 80)

print("""
📚 WEEK 14 COMPLETE — EVERYTHING WE BUILT:

  ─── Day 92: Docker Fundamentals ─────────────────────────────
  ✅ Containerization concepts: VMs vs containers
  ✅ Docker CLI, image layers, layer caching strategy
  ✅ Writing Dockerfiles: basic, Streamlit, multi-stage
  ✅ Docker Compose for multi-service ML apps
  ✅ Designed Security System container architecture

  ─── Day 93: Dockerize the Security System ───────────────────
  ✅ Dockerfile.api (multi-stage, FastAPI backend)
  ✅ Dockerfile.streamlit (single-stage, UI frontend)
  ✅ docker-compose.yml (FastAPI + Streamlit + Redis)
  ✅ .dockerignore (reduced build context 2.1GB → 45MB)
  ✅ config.py updated for full environment variable support

  ─── Day 94: RESTful API Design & FastAPI Deep Dive ──────────
  ✅ REST constraints and HTTP method semantics
  ✅ URL design: nouns, plural, kebab-case, /api/v1/ versioning
  ✅ FastAPI routers, lifespan context manager
  ✅ Dependency injection (model, Redis, auth, pagination)
  ✅ Middleware: request logging with timing and request ID
  ✅ Pydantic schemas with field validators
  ✅ Async patterns: await, run_in_executor, BackgroundTasks

  ─── Day 95: API Authentication & Security ───────────────────
  ✅ 7 common API attack vectors and defenses
  ✅ API key auth with SHA-256 hashing + timing-safe comparison
  ✅ JWT auth: access tokens (15min) + refresh tokens (7d)
  ✅ Role-based authorization: viewer → operator → admin
  ✅ SlowAPI rate limiting per endpoint + per API key
  ✅ Custom exception handlers for clean production errors
  ✅ Full security matrix for the Security System

  ─── Day 96: Building a CI/CD Pipeline ───────────────────────
  ✅ CI/CD concepts and ML-specific benefits
  ✅ GitHub Actions: triggers, jobs, steps, secrets, marketplace
  ✅ CI pipeline: lint (ruff + black) → test → docker build verify
  ✅ CD pipeline: docker push to Hub → SSH deploy + health check
  ✅ Docker image tagging: sha, latest, semver
  ✅ GitHub Actions cache (type=gha) for fast Docker builds

  ─── Day 97: Complete Test Suite & Performance ───────────────
  ✅ Advanced pytest: fixture scopes, teardown, session vs function
  ✅ Parametrized tests for boundary values and RBAC matrix
  ✅ Custom markers: unit, integration, auth, slow, performance
  ✅ pytest-cov: .coveragerc, 80% coverage target, HTML report
  ✅ Redis integration tests with fakeredis
  ✅ Locust load testing with weighted user tasks
  ✅ Load test result interpretation for ML APIs

  ─── Day 98: Model Monitoring & Logging ──────────────────────
  ✅ Structured JSON logging with Loguru
  ✅ Prometheus metrics: Counter, Gauge, Histogram
  ✅ ML-specific metrics: inference time, detections, confidence
  ✅ Full monitoring stack: Prometheus + Grafana + Alertmanager
  ✅ Alert rules: error rate, latency, model not loaded, slow inference
  ✅ Grafana dashboard panels for API health + ML performance
""")

print("\n✅ Exercise 4.1 Complete!")
print("=" * 80)


EXERCISE 4.1: Week 14 Complete Review

📚 WEEK 14 COMPLETE — EVERYTHING WE BUILT:

  ─── Day 92: Docker Fundamentals ─────────────────────────────
  ✅ Containerization concepts: VMs vs containers
  ✅ Docker CLI, image layers, layer caching strategy
  ✅ Writing Dockerfiles: basic, Streamlit, multi-stage
  ✅ Docker Compose for multi-service ML apps
  ✅ Designed Security System container architecture

  ─── Day 93: Dockerize the Security System ───────────────────
  ✅ Dockerfile.api (multi-stage, FastAPI backend)
  ✅ Dockerfile.streamlit (single-stage, UI frontend)
  ✅ docker-compose.yml (FastAPI + Streamlit + Redis)
  ✅ .dockerignore (reduced build context 2.1GB → 45MB)
  ✅ config.py updated for full environment variable support

  ─── Day 94: RESTful API Design & FastAPI Deep Dive ──────────
  ✅ REST constraints and HTTP method semantics
  ✅ URL design: nouns, plural, kebab-case, /api/v1/ versioning
  ✅ FastAPI routers, lifespan context manager
  ✅ Dependency injection (model, Redis, au

In [13]:
# ==================================================
# EXERCISE 4.2: WEEK 14 REFLECTION
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.2: Week 14 Reflection")
print("=" * 80)

print("""
💡 KEY LESSONS FROM WEEK 14:

  1. MLOps is engineering, not just ML
     The hardest parts of Week 14 weren\'t the ML — it was Docker
     networking, JWT token flows, CI/CD pipeline ordering, and
     Prometheus label cardinality. Production ML is 80% engineering.

  2. Containerization solves the reproducibility problem permanently
     After this week, "it works on my machine" is no longer acceptable.
     Every project should have a Dockerfile and docker-compose.yml.

  3. Security is a first-class concern, not an afterthought
     Adding auth, rate limiting, and exception handlers AFTER building
     the API would have been much harder. Design for security from day 1.

  4. Tests give you deployment confidence
     Without the test suite, every deployment is a gamble. With 80%+
     coverage and a CI pipeline, you KNOW the API works before it ships.

  5. Monitoring closes the feedback loop
     Writing code and shipping it is only half the job. Prometheus +
     Grafana + alerts tell you what\'s happening in production so you
     can catch degradation before users notice.

📊 WEEK 14 BY THE NUMBERS:

  Days completed:       7 / 7  ✅
  Notebooks written:    7
  Docker files created: 5  (Dockerfile.api, Dockerfile.streamlit,
                             docker-compose.yml, docker-compose.monitoring.yml,
                             .dockerignore)
  GitHub Actions files: 2  (ci.yml, cd.yml)
  Test files written:   6+
  Metrics defined:      8  (Counter, Gauge, Histogram)
  Alert rules:          5

🎯 WHAT\'S NEXT — WEEK 15 PREVIEW:

  Week 15: Advanced MLOps & Cloud Deployment
  ├── Day 99:  Cloud deployment fundamentals (AWS / GCP)
  ├── Day 100: Deploying to AWS EC2 + ECR
  ├── Day 101: Auto-scaling and load balancing
  ├── Day 102: Serverless ML inference (AWS Lambda)
  ├── Day 103: Model versioning with MLflow
  ├── Day 104: A/B testing ML models in production
  └── Day 105: Week 15 project — Cloud-deployed Security System
""")

print("\n✅ Exercise 4.2 Complete!")
print("=" * 80)


EXERCISE 4.2: Week 14 Reflection

💡 KEY LESSONS FROM WEEK 14:

  1. MLOps is engineering, not just ML
     The hardest parts of Week 14 weren't the ML — it was Docker
     networking, JWT token flows, CI/CD pipeline ordering, and
     Prometheus label cardinality. Production ML is 80% engineering.

  2. Containerization solves the reproducibility problem permanently
     After this week, "it works on my machine" is no longer acceptable.
     Every project should have a Dockerfile and docker-compose.yml.

  3. Security is a first-class concern, not an afterthought
     Adding auth, rate limiting, and exception handlers AFTER building
     the API would have been much harder. Design for security from day 1.

  4. Tests give you deployment confidence
     Without the test suite, every deployment is a gamble. With 80%+
     coverage and a CI pipeline, you KNOW the API works before it ships.

  5. Monitoring closes the feedback loop
     Writing code and shipping it is only half the job. P

In [14]:
print("\n" + "=" * 80)
print("🎉 DAY 98 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Structured JSON logging with Loguru (dev + production modes)
   ✅ Prometheus metrics: Counter, Gauge, Histogram for ML-specific tracking
   ✅ Full monitoring stack in docker-compose.monitoring.yml
   ✅ Alert rules for error rate, latency, model not loaded, slow inference
   ✅ Grafana dashboard panels for API health and ML performance
   ✅ Week 14 complete review and reflection

💡 KEY FILES CREATED TODAY:
   api/core/
   ├── logger.py               ← Loguru JSON logging setup
   └── metrics.py              ← Prometheus metrics definitions

   monitoring/
   ├── prometheus.yml          ← Scrape config + alert rules file
   ├── alerts.yml              ← 5 alert rules (error rate, latency, model)
   ├── alertmanager.yml        ← Notification routing
   └── grafana/
       ├── datasources/        ← Prometheus data source config
       └── dashboards/         ← Grafana dashboard JSON

   docker-compose.monitoring.yml  ← Prometheus + Grafana + Alertmanager

🎉 WEEK 14 COMPLETE!
   Full MLOps stack implemented:
   Docker → FastAPI → Auth → CI/CD → Testing → Monitoring

🎯 NEXT: Week 15 — Advanced MLOps & Cloud Deployment
   Starting Day 99: Cloud Deployment Fundamentals

Overall Progress: Day 98/168 (58.3% complete)
""")

print("=" * 80)
print("\n" + "=" * 80)
print("🎉 WEEK 14 COMPLETE — MLOPS & PRODUCTION ENGINEERING ✅")
print("=" * 80)


🎉 DAY 98 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Structured JSON logging with Loguru (dev + production modes)
   ✅ Prometheus metrics: Counter, Gauge, Histogram for ML-specific tracking
   ✅ Full monitoring stack in docker-compose.monitoring.yml
   ✅ Alert rules for error rate, latency, model not loaded, slow inference
   ✅ Grafana dashboard panels for API health and ML performance
   ✅ Week 14 complete review and reflection

💡 KEY FILES CREATED TODAY:
   api/core/
   ├── logger.py               ← Loguru JSON logging setup
   └── metrics.py              ← Prometheus metrics definitions

   monitoring/
   ├── prometheus.yml          ← Scrape config + alert rules file
   ├── alerts.yml              ← 5 alert rules (error rate, latency, model)
   ├── alertmanager.yml        ← Notification routing
   └── grafana/
       ├── datasources/        ← Prometheus data source config
       └── dashboards/         ← Grafana dashboard JSON

   docker-compose.monitoring.yml  ← Prometheus + Grafana + 